# 1 — Reformat Maize Tassels Counting UAV → crop-counter CVAT 1.1

Convert the **Maize Tassels Counting UAV** dataset
(`data/maize_tassels_counting_uav_dataset`) into the point-annotation layout that
`crop-counter` expects (see `crop-counter/examples/data`, same target used for the
wheat-head task in `D:/Projects/WheatHead`).

**Source format**
- `images/` — large UAV `.JPG` frames (`DJI_*.JPG`; some disambiguated on disk with a
  ` (2)` suffix).
- `labels/` — one VGG Image Annotator (VIA) CSV per image, columns
  `filename, file_size, file_attributes, region_count, region_id,
  region_shape_attributes, region_attributes`. Each row is one point annotation whose
  `region_shape_attributes` is JSON like `{"name":"point","cx":164,"cy":566}`. Files
  are UTF-8 with a BOM.
- `train.txt` / `val.txt` — the split lists. Each line is
  `\t`-separated: `/images/<name>.JPG\t/labels/<name>.csv`.

**Target format** — one folder per split (`train`, `val`), each holding `images/` and an
`annotations.xml` ("CVAT for images 1.1"). Every image is an `<image>` element carrying
one `<points label=... points="x,y" />` per annotation.

**Conversion rule** — the source is *already* point-annotated, so each VIA `point`
region maps straight to a CVAT `<points>` at `(cx, cy)`, labelled `Tassel`. Image
dimensions are read from the JPGs. Images with zero regions are written as `<image>`
elements with no points, so image counts are preserved. Only `train` and `val` exist
(no `test` split).

In [ ]:
from pathlib import Path
import csv
import json
import shutil
import xml.etree.ElementTree as ET

import pandas as pd
from PIL import Image

# --- paths ---------------------------------------------------------------
SRC = Path(r"path\to\MaizeTassel\data\maize_tassels_counting_uav_dataset")
DST = Path(r"path\to\MaizeTassel\data\maize_tassels_dataset_reformat")
SRC_IMAGES = SRC / "images"
SRC_LABELS = SRC / "labels"

# split list file  ->  crop-counter split folder
SPLITS = {
    "train.txt": "train",
    "val.txt": "val",
}

LABEL = "Tassel"      # every annotation is a maize tassel
COPY_IMAGES = True     # set False to (re)write only the XML and skip copying

assert SRC_IMAGES.is_dir(), f"missing image folder: {SRC_IMAGES}"
assert SRC_LABELS.is_dir(), f"missing label folder: {SRC_LABELS}"
DST.mkdir(parents=True, exist_ok=True)
print("source:", SRC)
print("target:", DST)

In [ ]:
def read_split_list(txt_path: Path):
    """Parse a split list (`train.txt` / `val.txt`).

    Each non-empty line is `\t`-separated `/images/<name>.JPG\t/labels/<name>.csv`.
    Returns a list of `(image_basename, label_basename)` pairs, using the on-disk
    basenames (which keep any ` (2)` disambiguation suffix).
    """
    pairs = []
    for raw in txt_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        fields = line.split("\t")
        if len(fields) != 2:
            raise ValueError(f"{txt_path.name}: expected 2 tab fields, got {fields!r}")
        img_name = Path(fields[0].strip()).name
        lbl_name = Path(fields[1].strip()).name
        pairs.append((img_name, lbl_name))
    return pairs


def via_csv_to_points(csv_path: Path):
    """Parse a VIA CSV into a list of (cx, cy) point annotations.

    Rows whose `region_shape_attributes` is a `point` contribute one (cx, cy).
    Empty / non-point rows (e.g. an image with `region_count` 0) are skipped, so a
    label-free image yields an empty list.
    """
    points = []
    # utf-8-sig strips the BOM that prefixes these VIA exports.
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            shape = (row.get("region_shape_attributes") or "").strip()
            if not shape or shape == "{}":
                continue
            attrs = json.loads(shape)
            if attrs.get("name") != "point":
                raise ValueError(
                    f"{csv_path.name}: unexpected shape {attrs.get('name')!r} "
                    f"(only 'point' supported)"
                )
            points.append((float(attrs["cx"]), float(attrs["cy"])))
    return points


def build_cvat_xml(records):
    """Build a CVAT-1.1 <annotations> tree from (name, width, height, points)."""
    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"
    for idx, (name, width, height, points) in enumerate(records):
        image_el = ET.SubElement(
            root, "image",
            id=str(idx), name=name, width=str(width), height=str(height),
        )
        for (x, y) in points:
            ET.SubElement(
                image_el, "points",
                label=LABEL, occluded="0",
                points=f"{x:.2f},{y:.2f}", z_order="0",
            )
    ET.indent(root, space="  ")  # pretty-print (Python 3.9+)
    return ET.ElementTree(root)

In [ ]:
def reformat_split(txt_name: str, split: str):
    """Convert one split list into a crop-counter split folder."""
    pairs = read_split_list(SRC / txt_name)
    out_dir = DST / split
    out_images = out_dir / "images"
    out_images.mkdir(parents=True, exist_ok=True)

    records = []
    n_points = 0
    n_empty = 0
    for i, (img_name, lbl_name) in enumerate(pairs, start=1):
        src_img = SRC_IMAGES / img_name
        src_lbl = SRC_LABELS / lbl_name
        if not src_img.is_file():
            raise FileNotFoundError(f"{txt_name}: image not found -> {src_img}")
        if not src_lbl.is_file():
            raise FileNotFoundError(f"{txt_name}: label not found -> {src_lbl}")

        with Image.open(src_img) as im:
            width, height = im.size

        points = via_csv_to_points(src_lbl)
        n_points += len(points)
        n_empty += (len(points) == 0)
        records.append((img_name, width, height, points))

        if COPY_IMAGES:
            shutil.copy2(src_img, out_images / img_name)

        if i % 50 == 0:
            print(f"  {split}: {i}/{len(pairs)} images processed")

    xml_path = out_dir / "annotations.xml"
    build_cvat_xml(records).write(xml_path, encoding="utf-8", xml_declaration=True)

    print(
        f"[{split}] {len(records)} images, {n_points} points, "
        f"{n_empty} empty  ->  {xml_path}"
    )
    return {"split": split, "images": len(records), "points": n_points, "empty": n_empty}


summary = [reformat_split(txt, split) for txt, split in SPLITS.items()]
pd.DataFrame(summary)

## Verify

Round-trip the output through `crop-counter`'s own CVAT loader and check that the
reloaded point counts match what we wrote. The default label filter is wheat-specific
(`COUNTED_LABELS = ("Wheat", "Volunteer")`), so pass `labels=None` to keep every point
regardless of its `Tassel` label.

In [ ]:
import sys

sys.path.insert(0, str(Path(r"D:\Projects\WheatHead\crop-counter\src")))
from cropcounter.crop_dataset import parse_cvat_1_1  # noqa: E402

for row in summary:
    split = row["split"]
    recs = parse_cvat_1_1(DST / split / "annotations.xml", labels=None)
    total = sum(len(r.points) for r in recs)
    ok = (len(recs) == row["images"]) and (total == row["points"])
    print(
        f"[{split}] reloaded {len(recs)} images / {total} points  "
        f"{'OK' if ok else 'MISMATCH'}"
    )
    assert ok, f"round-trip mismatch for split={split}"

In [ ]:
# Peek at the first image element of the train split
print((DST / "train" / "annotations.xml").read_text(encoding="utf-8")[:900])